# Readme E-Rara

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-Rara-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .



## Basiskonfiguration


Die E-Rara-Signaturen setzen sich aus dem E-Rara DOI zusammen. Im Config-File wird die Abteilung hinzugefügt, das Script ergänzt den DOI. 


### Config.py

Beispieldaten für ZHB E-Rara. Anpassungen können in der config.py vorgenommen werden. 

    user = 'name of person ingesting this object'
    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Rara'
    collection_id = 'zhb_erara'
    last_changed = 'yyyy-mm-dd'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    signature = 'zhb_'
    
### OAI configuration 

Für die E-Rara-Bestände wird die Zenodo-OAI-Schnittstelle verwendet, da hier die Original-ZIP-Kapseln liegen und auf alle andern Systeme verlinkt wird (Alma, E-Rara).

Zenodo: siehe https://developers.zenodo.org/#oai-pmh

    set name: lara_e-rara
    listrecords: user-lara_e-rara

### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format info/signature.json.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory 'fulldum' als menschenlesbarer Nachweis, welche Datenobjekte eingelagert wurden. 

### Marcxml aus Alma (SRU)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/signature.xml


### Ablage Datenobjekte 
Die E-Rara-Zipkapseln der ZHB sind nach folgender Struktur benannt:
DOI (Punkt/Schrägstrich ersetzt durch Unterstrich) _ (Digitalisierungsdatum/Workflow?) _ master _ version . zip

Beispiel:

     10_3931_e-rara-86416_20201027T114533_master_ver1.zip

Hier wird vorausgesetzt, dass die Daten mit dieser Bezeichnung im Ordner objects liegen.
Der DOI wird in dieser Form (mit Unterstrichen) im Feld "additional" gespeichert. 


### TODO Create Befehl für gocfl generieren

Die gocfl create Befehle für die E-Rara-Sammlung (alle Objekte) werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt mit der Signature. 

Muster:

    gocf create ./archiv.zip ./object metadata:./metadata --config ./config/gocfl.toml -i 'signature'  --ext-NNNN-metafile-source ./info.json
    


In [2]:
from sickle import Sickle
import json
import config
import xml.etree.ElementTree as ET
import requests
import pandas as pd


# Initialize the client by passing the base URL and fetch records from the OAI Set

base_url = 'https://zenodo.org/oai2d'
prefix = 'oai_dc'
set_name = 'user-lara_e-rara'

sickle = Sickle(base_url)
records = sickle.ListRecords(metadataPrefix=prefix, set=set_name)
record = records.next()

completed_iterating = False
recordCount = 1
completeSet = []

# Iterate through records and collect metadata info for json export
# Uncomment one of the next 2 lines for testing/production:

#while not completed_iterating:
while recordCount < 5:
    try:
        
        infoSet = {}                
        infoSet["additional"] = ''
        infoSet["address"] = config.address
        infoSet["collection"] = config.collection
        infoSet["collection_id"] = config.collection_id
        infoSet["created"] = record.metadata["date"][0]
        infoSet["identifiers"] = record.metadata["identifier"]
        infoSet["ingest_workflow"] = config.ingest_workflow
        infoSet["keywords"] = config.keywords
        infoSet["last_changed"] = config.last_changed
        infoSet["organisation"] = config.organisation
        infoSet["organisation_id"] = config.organisation_id
        infoSet["references"] = record.metadata["relation"]
        infoSet["sets"] = config.sets
        infoSet["signature"] = ''
        infoSet["title"] = record.metadata["title"][0]
        infoSet["user"] = config.user                # print(infoSet["references"])
        
        # add e-rara doi and alma id to identifiers:
        erara_doi = ''
        alma_id = ''
        
        for i in infoSet["references"]:
            if i.startswith('doi:10.3931/e-rara'):
                erara_doi = i[4:]         
            
            if i.startswith('url:https://rzs') or i.startswith('url:https://swisscovery'):                
                alma_id = i.partition('alma')[2]              

        print("E-Rara-DOI:", erara_doi, "| MMS-ID:", alma_id)     
        infoSet["identifiers"].insert(0, erara_doi)
        infoSet["identifiers"].insert(0, str(alma_id))         #        print(infoSet["identifiers"])        
        
        # save beginning of file name to 'additional'
        erara_filename = erara_doi.replace('.','_').replace('/','_') #         print(erara_filename)
        infoSet["additional"] = erara_filename
        
        #create signature:
        signature = config.signature+erara_filename
        infoSet["signature"] = signature
        print("Signature:",signature)        #debugging:        print(infoSet)
        
        # prepare filename for json export
        info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
        infofile = f"info/{signature}.json"
        with open(infofile, "w") as outfile:
            outfile.write(info_json)
            print(f"---\ninfo.json saved as {infofile}")
        
        # add info to completeSet
        completeSet.append(infoSet)
       
        # get metadata from Alma OAI as MARCXML
        
        sru_url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
        query = f"{sru_url}?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id={alma_id}"
        response = requests.get(query)
        if response.status_code != 200:
            raise Exception(f"SRU request failed with status code {response.status_code}")

        # Save the response content (MARCXML) to a file
        metafile = f"metadata/{signature}.xml"
        with open(metafile, 'wb') as file:
            file.write(response.content)
            print(f"---\nRecord with ID {alma_id} saved as {metafile}\n---")    
            
        # write signature to file
        sigfile = "fulldump/signatures.txt"
        with open(sigfile, 'a') as file:
            file.write(signature)
            file.write("\n")
            print(f"signatures appended to {sigfile}\n")

        #continue with next record
        recordCount = recordCount +1
        record = records.next()
        
    except StopIteration:
        completed_iterating = True

# Writing completeSet as json file
fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "fulldump/erara_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nfulldump written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "fulldump/erara_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Saved fulldump in excel file as {fullexcelfile}")


E-Rara-DOI: 10.3931/e-rara-86408 | MMS-ID: 995739210105505
Signature: zhb_10_3931_e-rara-86408
---
info.json saved as info/zhb_10_3931_e-rara-86408.json
---
Record with ID 995739210105505 saved as metadata/zhb_10_3931_e-rara-86408.xml
---
signatures written to fulldump/signatures.txt

E-Rara-DOI: 10.3931/e-rara-86373 | MMS-ID: 993029620105505
Signature: zhb_10_3931_e-rara-86373
---
info.json saved as info/zhb_10_3931_e-rara-86373.json
---
Record with ID 993029620105505 saved as metadata/zhb_10_3931_e-rara-86373.xml
---
signatures written to fulldump/signatures.txt

E-Rara-DOI: 10.3931/e-rara-86409 | MMS-ID: 997037850105505
Signature: zhb_10_3931_e-rara-86409
---
info.json saved as info/zhb_10_3931_e-rara-86409.json
---
Record with ID 997037850105505 saved as metadata/zhb_10_3931_e-rara-86409.xml
---
signatures written to fulldump/signatures.txt

E-Rara-DOI: 10.3931/e-rara-87374 | MMS-ID: 994403000105505
Signature: zhb_10_3931_e-rara-87374
---
info.json saved as info/zhb_10_3931_e-rara-